# SemMap · Исправленный Colab-пайплайн: тексты → ConceptNet → Parquet

**Чистая версия без monkey-patch, под Python 3.12+, T4/L4.**

Данные:
- [Txt.zip](https://drive.google.com/file/d/1I8dYdI4eLVmlhZjNewxl63EGgJKLmDqR/view) — исходные тексты на Google Drive.
- [cstr/conceptnet-normalized-multi](https://huggingface.co/datasets/cstr/conceptnet-normalized-multi) — SQLite ConceptNet 5.5, автоматически скачивается на локальный SSD Colab.
- [Папка результатов](https://drive.google.com/drive/folders/1C7r8A0L0afrfq_8KTebOBM4BJqbZjXM4) — отдельная папка `text-corpus-<EXPERIMENT_LABEL>`.

**Исправлено:** работа с нормализованной SQLite как с колонкой `language`, так и без неё; восстановление после сбоя при существующем `progress.json`; безопасная сегментированная лемматизация spaCy для документов длиной в миллионы символов без превышения E088; подробные логи ошибок из дочернего процесса. **Никаких переопределений методов SQLite, pathlib или модулей в ядре Colab нет.**

Версия учитывает только **английский язык**. Подготовка текстовых графов использует CPU; T4/L4 можно затем задействовать для численных алгоритмов semgraphex. Объём данных ограничивается `MAX_DOCS`, без скрытого отсечения рёбер.

Для повторного запуска оставляйте прежние входные данные и **ту же метку**. Если старый эксперимент создан до исправления исходников, сохранённые графы остаются на Drive, но при несовместимом `run.json` будет предложено начать новую метку.

**Важно:** при повторном запуске с той же меткой и неизменным `run.json` обработанные документы сохраняются. Если изменился SHA исходного кода в контракте, существующие опубликованные графы нельзя автоматически смешивать с новой версией; следует использовать новую метку. Если нет `run.json` (как при ошибке spaCy до первого документа), старую метку можно оставить.


In [ ]:
# Основные параметры
EXPERIMENT_LABEL = "semmap-text-hf55-spacy-capped-01"
SOURCE_ZIP_ID = "1I8dYdI4eLVmlhZjNewxl63EGgJKLmDqR"
RESULT_PARENT_ID = "1C7r8A0L0afrfq_8KTebOBM4BJqbZjXM4"
HF_DATASET_REVISION = "4d3fd17e80543e88d9823e2f4fe47901dcf9a4e6"

REPO_BRANCH = "feature/text-corpus-colab-drive-resume"
REPO_DIR = "/content/semgraphex-text-clean"  # отдельная чистая копия кода
WORKSPACE = "/content/semmap-text-corpus"     # вычисления и кэш БД на SSD
MAX_DOCS = 100                                # None: весь ZIP
WORKERS = 0                                   # 0: автоматический подбор
THREADS_PER_WORKER = 1
MEMORY_LIMIT_PER_WORKER = "1GB"
MAX_EDGES_PER_DOC = None               # нет ограничения на весь граф; запись разбивается
CHUNKED_GRAPH = True                  # последовательные части, ОДИН общий граф
TEXT_CHUNK_CHARS = 100_000            # кодовые точки Unicode, границы слов сохраняются
EDGE_PARTITION_ROWS = 1_000_000       # не более 1 млн рёбер на один Parquet-файл
MAX_TOTAL_EDGES_PER_DOC = 150_000_000 # сохранять не более 150 млн рёбер; предупредить и продолжить
SPACY_MODEL = "en_core_web_sm"  # None: без spaCy


In [ ]:
from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()

from pathlib import Path
from googleapiclient.discovery import build
import google.auth
import os, sys, subprocess, json, shutil

credentials, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=credentials, cache_discovery=False)
root_id = drive_api.files().get(fileId="root", fields="id").execute()["id"]

def mounted_drive_path(file_id):
    """Find a Drive file via verified parent IDs; no hardcoded MyDrive subpaths."""
    pieces, current = [], file_id
    for _ in range(40):
        if current == root_id:
            return Path("/content/drive/MyDrive").joinpath(*reversed(pieces))
        info = drive_api.files().get(
            fileId=current, fields="id,name,parents"
        ).execute()
        pieces.append(info["name"])
        parents = info.get("parents") or []
        if len(parents) != 1:
            raise RuntimeError(
                f"Файл {file_id} не имеет одного доступного родителя в MyDrive"
            )
        current = parents[0]
    raise RuntimeError("Drive parent chain too deep")

archive_path = mounted_drive_path(SOURCE_ZIP_ID)
result_parent_path = mounted_drive_path(RESULT_PARENT_ID)
if not archive_path.is_file():
    raise FileNotFoundError(f"Исходный ZIP недоступен: {archive_path}")
if not result_parent_path.is_dir():
    raise FileNotFoundError(f"Папка результатов недоступна: {result_parent_path}")
print("Исходный ZIP:", archive_path)
print("Результаты:", result_parent_path / ("text-corpus-" + EXPERIMENT_LABEL))


In [ ]:
import platform
import psutil
import re

cpu_count = os.cpu_count() or 2
available_ram = psutil.virtual_memory().available
free_disk = shutil.disk_usage("/content").free
workers = WORKERS or max(1, min(12, cpu_count - 1, int(available_ram // (2 * 1024**3))))
print("Python:", platform.python_version())
print(f"CPU: {cpu_count}, RAM available: {available_ram / 2**30:.1f} GiB")
print(f"Local disk free: {free_disk / 2**30:.1f} GiB")
if sys.version_info < (3, 12):
    raise RuntimeError("Нужен Colab Python 3.12+. Измените runtime.")
if free_disk < 4_000_000_000:
    raise RuntimeError("Недостаточно SSD для SQLite (~1.78 GB), ZIP и результатов.")
# DuckDB memory_limit is PER WORKER. spaCy, Python, SQLite and ZIP need
# additional RAM; requested workers × threads must not overcommit Colab.
memory_match = re.fullmatch(r"(?i)\s*(\d+(?:\.\d+)?)\s*(GB|GIB|MB|MIB)\s*", MEMORY_LIMIT_PER_WORKER)
if not memory_match:
    raise ValueError("MEMORY_LIMIT_PER_WORKER: ожидается '1GB', '2GB', '512MB' и т.п.")
amount, unit = float(memory_match.group(1)), memory_match.group(2).upper()
multiplier = 1024**3 if unit in ("GB", "GIB") else 1024**2
duckdb_budget = amount * multiplier
extra_per_worker = (1.0 if SPACY_MODEL else .5) * 1024**3
runtime_reserve = 2 * 1024**3
max_cpu = max(1, cpu_count // THREADS_PER_WORKER)
max_ram = max(1, int((available_ram - runtime_reserve) //
                     (duckdb_budget + extra_per_worker)))
safe_workers = max(1, min(workers, max_cpu, max_ram))
if safe_workers < workers:
    print(f"[safety] Ограничение параллельности: {workers} → {safe_workers}. "
          f"Указано {MEMORY_LIMIT_PER_WORKER}/worker + память spaCy, "
          f"{THREADS_PER_WORKER} CPU threads/worker.")
    workers = safe_workers
print(f"Selected workers: {workers} × {THREADS_PER_WORKER} thread(s)")
try:
    import torch
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("GPU не назначен (построение графов выполняется на CPU)")
except ImportError:
    print("GPU диагностика недоступна; построение графов всё равно CPU")


In [ ]:
# Клонируем исходники в ОТДЕЛЬНЫЙ рабочий каталог без старых патчей.
repo = Path(REPO_DIR)
if not (repo / ".git").is_dir():
    if repo.exists():
        raise FileExistsError(f"{repo} существует, но не является Git-репозиторием")
    subprocess.run([
        "git", "clone", "--single-branch", "--branch", REPO_BRANCH,
        "https://github.com/SemanticMap/semgraphex.git", str(repo)
    ], check=True)
else:
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=repo, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=repo, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/" + REPO_BRANCH], cwd=repo, check=True)

extras = "[text,text-lemma]" if SPACY_MODEL else "[text]"
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e", str(repo) + extras
], check=True)

revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print("Revision:", revision)
# Тестируем импорт в НОВОМ процессе: editable src-layout будет корректно подключён.
subprocess.run([
    sys.executable, "-c",
    "import semmap_haken.text_colab as m; "
    "print('Import OK:', m.__file__)"
], check=True, cwd=repo)


In [ ]:
# Автоматическая установка и проверка spaCy (никаких monkey-patch).
if SPACY_MODEL:
    if SPACY_MODEL != "en_core_web_sm":
        raise ValueError("В текущем англоязычном режиме используйте en_core_web_sm")
    check_model = (
        "import spacy; p=spacy.load('en_core_web_sm'); "
        "assert p('Cats are running.')[0].lemma_.lower() == 'cat'; "
        "print('spaCy + en_core_web_sm OK')"
    )
    first_check = subprocess.run([sys.executable, "-c", check_model], cwd=repo)
    if first_check.returncode:
        subprocess.run([
            sys.executable, "-m", "spacy", "download", SPACY_MODEL
        ], check=True, cwd=repo)
        subprocess.run([sys.executable, "-c", check_model], check=True, cwd=repo)
else:
    print("spaCy выключена: остаётся точное сопоставление слов.")


In [ ]:
# Скачиваем фиксированную ревизию HF SQLite на локальный SSD.
# Схема проверяется БЕЗ динамического изменения функций.
hub_code = """
from semmap_haken.text_hf import download_database
import json, sys
info = download_database(
    cache_dir=sys.argv[1], revision=sys.argv[2]
)
print(json.dumps(info, ensure_ascii=False, indent=2))
"""
hub_cache = str(Path(WORKSPACE) / "hf-cache")
subprocess.run([
    sys.executable, "-c", hub_code, hub_cache, HF_DATASET_REVISION
], check=True, cwd=repo)


## Прогресс выполнения

Во время работы в этой ячейке выводится обновляемая сводка, например:

```
[progress] Документы: 12/100 (12.0%) | article_01.txt: 42% (поиск понятий ConceptNet, 42000/100000) | article_02.txt: 85% (запись рёбер в Parquet)
```

- **12/100 (12%)** — точное число документов, полностью обработанных и опубликованных в Google Drive.
- **42%** — оценка готовности *конкретного документа* по этапам; число `42000/100000` — фактический прогресс по словам/символам, когда он доступен.
- На длительном SQL-запросе процент может временно оставаться прежним; это не зависание процесса. Стадия показывает, что именно выполняется.
- Сводка обновляется примерно раз в 10 секунд и сохраняется в `progress.json` папки эксперимента, даже до завершения документа.

Параллельно обрабатываются несколько текстов; прогресс выводится по каждому активному документу.

**Разбиение длинных документов:** один текст обрабатывается последовательно фрагментами до `TEXT_CHUNK_CHARS` символов; слова получают глобальные IDs, все фрагменты вместе образуют **один граф документа**. Для рёбер создаётся несколько `edges_part_*.parquet` с максимумом `EDGE_PARTITION_ROWS` на файл; каждый источник сравнивается со **всеми** словами документа, включая другие фрагменты. `open_corpus()` читает части как единую таблицу `edges`. `MAX_EDGES_PER_DOC=None` снимает старый лимит 20 млн для всего графа; независимый `MAX_TOTAL_EDGES_PER_DOC` — **предел числа сохраняемых рёбер в одном графе**. При превышении будет предупреждение, файл сохранится как неполный (`graph_complete=false`), а обработка следующих документов продолжится. Число пропущенных рёбер записывается в `manifest.json` и `progress.json`. Выбираются первые N рёбер по исходным идентификаторам слов, затем целям и утверждениям ConceptNet — это не случайная выборка, что важно учитывать при исследовании.


In [ ]:
from collections import deque

cmd = [
    sys.executable, "-u", "-m", "semmap_haken.text_colab",
    "--archive", str(archive_path),
    "--hf-db",
    "--hf-revision", HF_DATASET_REVISION,
    "--drive-parent", str(result_parent_path),
    "--label", EXPERIMENT_LABEL,
    "--workspace", WORKSPACE,
    "--workers", str(workers),
    "--threads-per-worker", str(THREADS_PER_WORKER),
    "--memory-limit", MEMORY_LIMIT_PER_WORKER,
    "--max-edges-per-document", str(MAX_EDGES_PER_DOC or 0),
]
if CHUNKED_GRAPH:
    cmd += ["--chunked-graph", "--chunk-chars", str(TEXT_CHUNK_CHARS),
            "--edge-partition-rows", str(EDGE_PARTITION_ROWS),
            "--max-total-edges-per-document", str(MAX_TOTAL_EDGES_PER_DOC)]
if MAX_DOCS is not None:
    cmd += ["--max-docs", str(MAX_DOCS)]
if SPACY_MODEL:
    cmd += ["--spacy-model", SPACY_MODEL]
print("Запуск:", " ".join(cmd), flush=True)

# Stream combined stdout/stderr, retain last diagnostic lines and inspect the
# persisted failure report. No kernel monkey patches or hidden subprocess errors.
tail = deque(maxlen=100)
with subprocess.Popen(cmd, cwd=repo, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True,
                      errors="replace", bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line)
    returncode = process.wait()
if returncode:
    progress_path = result_parent_path / ("text-corpus-" + EXPERIMENT_LABEL) / "progress.json"
    if progress_path.is_file():
        print("\n=== Ошибка из контрольной точки Drive ===", flush=True)
        try:
            print(json.dumps(json.loads(progress_path.read_text(encoding="utf-8")),
                             indent=2, ensure_ascii=False))
        except Exception as exc:
            print("Не удалось прочитать progress.json:", exc)
    print("\n=== Последние 100 строк stdout/stderr ===", flush=True)
    print("".join(tail), flush=True)
    raise RuntimeError(f"semmap_haken.text_colab завершился с кодом {returncode}. "
                       "Полная причина напечатана выше; прогресс на Drive сохранён.")


### При прерывании Colab

Перезапустите сеанс и выполните **Run all**, оставив прежнюю `EXPERIMENT_LABEL`, исходный ZIP, ревизию ConceptNet и модель spaCy. Скачивание Hugging Face повторится при новом VM, но завершённые документы будут восстановлены с Google Drive.

При смене версии исходного кода, модели или данных `run.json` может не совпасть — в этом случае используйте новую метку. **Не удаляйте старые папки** и не вставляйте патчи `Path.iterdir`, `sqlite3.connect` или `NormalizedConceptNet.__init__`.

**Поскольку изменились алгоритм и формат промежуточных результатов, используется новая метка эксперимента, чтобы не смешивать полные и ограниченные графы.** Старые папки на Drive не удаляются. Отдельный документ после публикации остаётся единой графовой структурой, даже если содержит десятки частей Parquet.
